In [1]:
import sys

print(sys.version)
print(sys.executable)

3.12.11 | packaged by conda-forge | (main, Jun  4 2025, 14:45:31) [GCC 13.3.0]
/data/ulead-36/Proyecto/.venv/bin/python


In [2]:
import flax
import inspect
from flax import nnx

print("Flax:", flax.__version__)
print(inspect.signature(nnx.Optimizer.update))

Flax: 0.12.8
(self, model: 'M', grads, /, **kwargs) -> 'optax.Updates'


In [3]:
# 1. Configuracion general
#
# Clasificador de color de diamante (D-N, 11 clases) a partir de la imagen.
# Framework: JAX + Flax NNX (consistente con la Tarea 2).
#
# Requiere que ya exista data/processed/diamonds_originales.h5, generado
# por extract.py, con los datasets: image_bytes, colour_label, y el
# atributo JSON label_mappings.
#
# NOTA DE DISENO: no se usa NINGUNA augmentation que altere el color
# (hue/saturacion/brillo) porque el color es la etiqueta que queremos
# predecir. Solo transformaciones geometricas serian seguras (no se
# incluyen aqui para mantener el primer entrenamiento simple).

from pathlib import Path
import json
import os
import sys
import time

import cv2
import h5py
import jax
import jax.numpy as jnp
import matplotlib

matplotlib.use("Agg")

import matplotlib.pyplot as plt
import numpy as np
import optax

from flax import nnx
from tqdm.auto import tqdm


PROJECT_ROOT = Path("/data/ulead-36/Proyecto")

PLOTS_OUTPUT_DIR = Path("/data/ulead-36/Proyecto/images")
PLOTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def mostrar(objeto):
    """Print tables in the console as a replacement for Jupyter mostrar()."""
    if hasattr(objeto, "to_string"):
        print(objeto.to_string())
    else:
        print(objeto)


def guardar_figura(nombre_archivo):
    ruta_salida = PLOTS_OUTPUT_DIR / nombre_archivo
    plt.savefig(ruta_salida, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Plot guardado: {ruta_salida}")


print(f"Raiz del proyecto: {PROJECT_ROOT}")
print(f"JAX devices: {jax.devices()}")


# 2. Parametros de ejecucion

RUTA_H5 = "/data/ulead-36/Proyecto/data/processed/diamonds_originales.h5"
TAMANO_IMAGEN = 96
TAMANO_LOTE = 64
NUM_EPOCAS = 20
TASA_APRENDIZAJE = 1e-3
PROPORCION_VALIDACION = 0.15
SEMILLA = 42


# 3. Cargador de datos
#
# Lee bytes de imagen crudos del HDF5 y los decodifica/redimensiona
# al vuelo (las imagenes se guardaron sin resize, con su formato original).

class DiamondColorDataset:
    """Envuelve el HDF5 de diamantes y expone lotes de imagenes + color."""

    def __init__(self, ruta_h5, tamano_imagen=96):
        self.ruta_h5 = ruta_h5
        self.tamano_imagen = tamano_imagen

        self._archivo = h5py.File(ruta_h5, "r")
        self.colour_label = self._archivo["colour_label"][:]
        self.image_bytes = self._archivo["image_bytes"]

        mapeos = json.loads(self._archivo.attrs["label_mappings"])
        self.colour_mapping = mapeos["colour"]
        self.num_clases = len(self.colour_mapping)

        self.indices_por_clase = {
            idx: np.where(self.colour_label == idx)[0]
            for idx in range(self.num_clases)
        }

    def __len__(self):
        return len(self.colour_label)

    def calcular_pesos_de_clase(self):
        conteos = np.array(
            [
                len(self.indices_por_clase[i])
                for i in range(self.num_clases)
            ],
            dtype=np.float64,
        )

        conteos = np.where(conteos == 0, 1, conteos)
        pesos = conteos.sum() / (self.num_clases * conteos)

        return pesos.astype(np.float32)

    def _decodificar_imagen(self, indice):
        bytes_crudos = self.image_bytes[indice]
        imagen = cv2.imdecode(bytes_crudos, cv2.IMREAD_COLOR)

        if imagen is None:
            return np.zeros(
                (
                    self.tamano_imagen,
                    self.tamano_imagen,
                    3,
                ),
                dtype=np.float32,
            )

        imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)

        imagen = cv2.resize(
            imagen,
            (self.tamano_imagen, self.tamano_imagen),
            interpolation=cv2.INTER_AREA,
        )

        return imagen.astype(np.float32) / 255.0

    def obtener_lote(self, indices):
        imagenes = np.stack(
            [
                self._decodificar_imagen(i)
                for i in indices
            ]
        )

        etiquetas = self.colour_label[indices]

        return imagenes, etiquetas

    def generar_lotes(
        self,
        indices_disponibles,
        tamano_lote,
        semilla=None,
        mezclar=True,
    ):
        rng = np.random.default_rng(semilla)
        orden = np.array(indices_disponibles)

        if mezclar:
            rng.shuffle(orden)

        for inicio in range(0, len(orden), tamano_lote):
            indices_lote = sorted(
                orden[inicio:inicio + tamano_lote].tolist()
            )

            # Evita batches minusculos e inestables en BatchNorm.
            if len(indices_lote) < 4:
                continue

            yield self.obtener_lote(indices_lote)

    def cerrar(self):
        self._archivo.close()


def dividir_train_val(
    dataset,
    proporcion_validacion,
    semilla,
):
    """
    Split estratificado por color: separa una proporcion de cada clase para
    validacion, para que las clases raras (K, L, M, N) tambien queden
    representadas en validacion.
    """
    rng = np.random.default_rng(semilla)

    indices_train = []
    indices_val = []

    for clase, indices in dataset.indices_por_clase.items():
        indices = indices.copy()
        rng.shuffle(indices)

        corte = max(
            1,
            int(len(indices) * proporcion_validacion),
        )

        indices_val.extend(indices[:corte].tolist())
        indices_train.extend(indices[corte:].tolist())

    return indices_train, indices_val


# 4. Modelo: CNN en Flax NNX
#
# Bloques conv -> batchnorm -> relu -> maxpool, seguidos de global average
# pooling (en vez de aplanar) para no depender de la posicion exacta del
# diamante en la foto.

class BloqueConvolucional(nnx.Module):
    def __init__(
        self,
        canales_entrada,
        canales_salida,
        *,
        rngs: nnx.Rngs,
    ):
        self.conv = nnx.Conv(
            canales_entrada,
            canales_salida,
            kernel_size=(3, 3),
            padding="SAME",
            rngs=rngs,
        )

        self.norma = nnx.BatchNorm(
            canales_salida,
            rngs=rngs,
        )

    def __call__(
        self,
        x,
        *,
        entrenando: bool,
    ):
        x = self.conv(x)

        x = self.norma(
            x,
            use_running_average=not entrenando,
        )

        x = nnx.relu(x)

        x = nnx.max_pool(
            x,
            window_shape=(2, 2),
            strides=(2, 2),
        )

        return x


class DiamondColorCNN(nnx.Module):
    def __init__(
        self,
        num_clases,
        *,
        rngs: nnx.Rngs,
        canales=(32, 64, 128, 256),
    ):
        self.bloque1 = BloqueConvolucional(
            3,
            canales[0],
            rngs=rngs,
        )

        self.bloque2 = BloqueConvolucional(
            canales[0],
            canales[1],
            rngs=rngs,
        )

        self.bloque3 = BloqueConvolucional(
            canales[1],
            canales[2],
            rngs=rngs,
        )

        self.bloque4 = BloqueConvolucional(
            canales[2],
            canales[3],
            rngs=rngs,
        )

        self.dropout = nnx.Dropout(
            rate=0.3,
            rngs=rngs,
        )

        self.salida = nnx.Linear(
            canales[3],
            num_clases,
            rngs=rngs,
        )

    def __call__(
        self,
        x,
        *,
        entrenando: bool = False,
    ):
        x = self.bloque1(
            x,
            entrenando=entrenando,
        )

        x = self.bloque2(
            x,
            entrenando=entrenando,
        )

        x = self.bloque3(
            x,
            entrenando=entrenando,
        )

        x = self.bloque4(
            x,
            entrenando=entrenando,
        )

        x = jnp.mean(
            x,
            axis=(1, 2),
        )

        x = self.dropout(
            x,
            deterministic=not entrenando,
        )

        return self.salida(x)


def calcular_loss(
    modelo,
    imagenes,
    etiquetas,
    pesos_de_clase,
    *,
    entrenando,
):
    logits = modelo(
        imagenes,
        entrenando=entrenando,
    )

    pesos_por_muestra = pesos_de_clase[etiquetas]

    loss_por_muestra = (
        optax.softmax_cross_entropy_with_integer_labels(
            logits,
            etiquetas,
        )
    )

    loss = jnp.mean(
        loss_por_muestra * pesos_por_muestra
    )

    exactitud = jnp.mean(
        jnp.argmax(logits, axis=-1) == etiquetas
    )

    return loss, exactitud


@nnx.jit
def paso_de_entrenamiento(
    modelo,
    optimizador,
    imagenes,
    etiquetas,
    pesos_de_clase,
):
    def funcion_de_perdida(modelo):
        return calcular_loss(
            modelo,
            imagenes,
            etiquetas,
            pesos_de_clase,
            entrenando=True,
        )

    (loss, exactitud), gradientes = nnx.value_and_grad(
        funcion_de_perdida,
        has_aux=True,
    )(modelo)

    # Flax 0.11+ requiere recibir el modelo y los gradientes.
    optimizador.update(modelo, gradientes)

    return loss, exactitud


@nnx.jit
def paso_de_evaluacion(
    modelo,
    imagenes,
    etiquetas,
    pesos_de_clase,
):
    return calcular_loss(
        modelo,
        imagenes,
        etiquetas,
        pesos_de_clase,
        entrenando=False,
    )


# 5. Entrenamiento

print("Cargando dataset desde:", RUTA_H5)

dataset = DiamondColorDataset(
    RUTA_H5,
    tamano_imagen=TAMANO_IMAGEN,
)

print(f"Total de imagenes: {len(dataset)}")
print(f"Clases de color: {dataset.colour_mapping}")

pesos_de_clase = jnp.array(
    dataset.calcular_pesos_de_clase()
)

print(
    f"Pesos de clase (por desbalance): "
    f"{pesos_de_clase}"
)

indices_train, indices_val = dividir_train_val(
    dataset,
    PROPORCION_VALIDACION,
    SEMILLA,
)

print(
    f"Imagenes de entrenamiento: "
    f"{len(indices_train)}"
)

print(
    f"Imagenes de validacion:    "
    f"{len(indices_val)}"
)

rngs = nnx.Rngs(SEMILLA)

modelo = DiamondColorCNN(
    num_clases=dataset.num_clases,
    rngs=rngs,
)

optimizador = nnx.Optimizer(
    modelo,
    optax.adamw(TASA_APRENDIZAJE),
    wrt=nnx.Param,
)

historial_train = []
historial_val = []

mejor_loss_val = float("inf")
inicio = time.time()

num_lotes_train = len(indices_train) // TAMANO_LOTE

if len(indices_train) % TAMANO_LOTE >= 4:
    num_lotes_train += 1

num_lotes_val = len(indices_val) // TAMANO_LOTE

if len(indices_val) % TAMANO_LOTE >= 4:
    num_lotes_val += 1


for epoca in range(NUM_EPOCAS):
    inicio_epoca = time.time()

    perdidas_epoca = []
    exactitudes_epoca = []

    barra_train = tqdm(
        dataset.generar_lotes(
            indices_train,
            TAMANO_LOTE,
            semilla=epoca,
        ),
        total=num_lotes_train,
        desc=(
            f"Epoca {epoca + 1}/{NUM_EPOCAS} "
            f"- entrenamiento"
        ),
        unit="lote",
    )

    for imgs, labels in barra_train:
        loss, exactitud = paso_de_entrenamiento(
            modelo,
            optimizador,
            jnp.array(imgs),
            jnp.array(labels),
            pesos_de_clase,
        )

        perdidas_epoca.append(float(loss))
        exactitudes_epoca.append(float(exactitud))

        barra_train.set_postfix(
            loss=f"{np.mean(perdidas_epoca):.4f}",
            acc=f"{np.mean(exactitudes_epoca):.3f}",
        )

    loss_train = float(
        np.mean(perdidas_epoca)
    )

    exactitud_train = float(
        np.mean(exactitudes_epoca)
    )

    perdidas_val = []
    exactitudes_val = []

    barra_val = tqdm(
        dataset.generar_lotes(
            indices_val,
            TAMANO_LOTE,
            mezclar=False,
        ),
        total=num_lotes_val,
        desc=(
            f"Epoca {epoca + 1}/{NUM_EPOCAS} "
            f"- validacion"
        ),
        unit="lote",
    )

    for imgs, labels in barra_val:
        loss, exactitud = paso_de_evaluacion(
            modelo,
            jnp.array(imgs),
            jnp.array(labels),
            pesos_de_clase,
        )

        perdidas_val.append(float(loss))
        exactitudes_val.append(float(exactitud))

        barra_val.set_postfix(
            loss=f"{np.mean(perdidas_val):.4f}",
            acc=f"{np.mean(exactitudes_val):.3f}",
        )

    loss_val = float(
        np.mean(perdidas_val)
    )

    exactitud_val = float(
        np.mean(exactitudes_val)
    )

    historial_train.append(
        {
            "loss": loss_train,
            "exactitud": exactitud_train,
        }
    )

    historial_val.append(
        {
            "loss": loss_val,
            "exactitud": exactitud_val,
        }
    )

    marca = (
        " *"
        if loss_val < mejor_loss_val
        else ""
    )

    mejor_loss_val = min(
        mejor_loss_val,
        loss_val,
    )

    tiempo_epoca = (
        time.time() - inicio_epoca
    )

    print(
        f"Epoca {epoca + 1}/{NUM_EPOCAS} | "
        f"train loss={loss_train:.4f} "
        f"acc={exactitud_train:.3f} | "
        f"val loss={loss_val:.4f} "
        f"acc={exactitud_val:.3f} | "
        f"tiempo={tiempo_epoca / 60:.2f} min"
        f"{marca}"
    )


print(
    f"Tiempo total de entrenamiento: "
    f"{(time.time() - inicio) / 60:.2f} minutos"
)


# 6. Graficar curvas de entrenamiento

fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(12, 5),
)

ax1.plot(
    [h["loss"] for h in historial_train],
    label="Train",
)

ax1.plot(
    [h["loss"] for h in historial_val],
    label="Validacion",
)

ax1.set_title("Loss por epoca")
ax1.set_xlabel("Epoca")
ax1.set_ylabel("Loss (cross-entropy ponderada)")
ax1.legend()

ax2.plot(
    [h["exactitud"] for h in historial_train],
    label="Train",
)

ax2.plot(
    [h["exactitud"] for h in historial_val],
    label="Validacion",
)

ax2.set_title("Exactitud por epoca")
ax2.set_xlabel("Epoca")
ax2.set_ylabel("Exactitud")
ax2.legend()

plt.tight_layout()

guardar_figura(
    "curvas_entrenamiento_color.png"
)


# 7. Matriz de confusion en el set de validacion

matriz_confusion = np.zeros(
    (
        dataset.num_clases,
        dataset.num_clases,
    ),
    dtype=int,
)

for imgs, labels in dataset.generar_lotes(
    indices_val,
    TAMANO_LOTE,
    mezclar=False,
):
    logits = modelo(
        jnp.array(imgs),
        entrenando=False,
    )

    predicciones = np.array(
        jnp.argmax(logits, axis=-1)
    )

    for real, pred in zip(
        labels,
        predicciones,
    ):
        matriz_confusion[real, pred] += 1


nombres_clases = [
    clase
    for clase, indice in sorted(
        dataset.colour_mapping.items(),
        key=lambda elemento: elemento[1],
    )
]

fig, ax = plt.subplots(
    figsize=(8, 7)
)

im = ax.imshow(
    matriz_confusion,
    cmap="Blues",
)

ax.set_xticks(
    range(dataset.num_clases)
)

ax.set_yticks(
    range(dataset.num_clases)
)

ax.set_xticklabels(
    nombres_clases
)

ax.set_yticklabels(
    nombres_clases
)

ax.set_xlabel("Prediccion")
ax.set_ylabel("Color real")

ax.set_title(
    "Matriz de confusion - "
    "color de diamante (validacion)"
)

for i in range(dataset.num_clases):
    for j in range(dataset.num_clases):
        ax.text(
            j,
            i,
            matriz_confusion[i, j],
            ha="center",
            va="center",
            fontsize=8,
        )

plt.colorbar(
    im,
    ax=ax,
)

plt.tight_layout()

guardar_figura(
    "matriz_confusion_color.png"
)

dataset.cerrar()

print("Listo.")

/data/ulead-36/Proyecto/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Raiz del proyecto: /data/ulead-36/Proyecto
JAX devices: [CpuDevice(id=0)]
Cargando dataset desde: /data/ulead-36/Proyecto/data/processed/diamonds_originales.h5
Total de imagenes: 47684
Clases de color: {'D': 0, 'E': 1, 'F': 2, 'G': 3, 'H': 4, 'I': 5, 'J': 6, 'K': 7, 'L': 8, 'M': 9, 'N': 10}
Pesos de clase (por desbalance): [ 0.74830127  0.6985029   0.62706625  0.5688857   0.66598696  0.823501
  1.0243169   1.6659912   3.15037     5.756851   10.420455  ]
Imagenes de entrenamiento: 40538
Imagenes de validacion:    7146


Epoca 1/20 - validacion: 100%|██████████| 112/112 [01:21<00:00,  1.37lote/s, acc=0.122, loss=6.6623]


Epoca 1/20 | train loss=1.7103 acc=0.269 | val loss=6.6623 acc=0.122 | tiempo=8.85 min *


Epoca 2/20 - validacion: 100%|██████████| 112/112 [01:03<00:00,  1.75lote/s, acc=0.092, loss=3.9707]


...

Epoca 18/20 | train loss=1.1329 acc=0.466 | val loss=2.1822 acc=0.296 | tiempo=6.73 min


Epoca 19/20 - validacion: 100%|██████████| 112/112 [01:22<00:00,  1.35lote/s, acc=0.036, loss=3.9333]


Epoca 19/20 | train loss=1.1114 acc=0.474 | val loss=3.9333 acc=0.036 | tiempo=8.69 min


Epoca 20/20 - validacion: 100%|██████████| 112/112 [01:04<00:00,  1.74lote/s, acc=0.015, loss=10.2524]


Epoca 20/20 | train loss=1.0880 acc=0.482 | val loss=10.2524 acc=0.015 | tiempo=7.48 min
Tiempo total de entrenamiento: 140.57 minutos
Plot guardado: /data/ulead-36/Proyecto/images/curvas_entrenamiento_color.png
Plot guardado: /data/ulead-36/Proyecto/images/matriz_confusion_color.png
Listo.
